# Measurement and Analysis Notebook

This notebook provides a structured and reusable framework for documenting the experimental measurement workflow and the associated analysis steps.

## Purpose
The notebook is intended to:
- document the experimental setup and instrument configuration,
- define the measurement workflow for resonator characterization,
- record sweep parameters and auxiliary control settings,

## Typical contents
The notebook may include:
1. measurement setup overview  
2. instrument initialization and configuration  
3. VNA sweep and overview measurements  
4. current / flux sweep routines  


<table>
<thead>
  <tr>
    <th>Device / Chip</th>
    <th>Sample Holder</th>
    <th>Input Line</th>
    <th>Output Line</th>
    <th>HEMT Amplifier</th>
    <th>Input Attenuation Chain</th>
    <th>DC Wiring</th>
    <th>VNA</th>
    <th>Microwave Source</th>
  </tr>
</thead>
<tbody>
  <tr>
    <td>[Device ID]</td>
    <td>[Holder Description]</td>
    <td>[Input Path Description]</td>
    <td>[Output Path Description]</td>
    <td>[Amplifier Model / Notes]</td>
    <td>[x dB @ RT] - [x dB @ 4K] - [x dB @ Still] - [x dB @ MXC]</td>
    <td>[DC Line Count / Type]</td>
    <td>[Instrument Model]</td>
    <td>[Source Model]</td>
  </tr>
</tbody>
</table>

<table>
<thead>
  <tr>
    <th>Resonator<br></th>
    <th>Frequency / GHz</th>
    <th>$Q_{l}$</th>
    <th>$Q_{c}$</th>
    <th>$Q_{i}$</th>
  </tr>
</thead>
<tbody>
  <tr>
    <td>1</td>
    <td></td>
    <td></td>
    <td></td>
    <td></td>
  </tr>
  <tr>
    <td>2</td>
    <td></td>
    <td></td>
    <td></td>
    <td></td>
  </tr>
  <tr>
    <td>3</td>
    <td></td>
    <td></td>
    <td></td>
    <td></td>
  </tr>  
  <tr>
    <td>4</td>
    <td></td>
    <td></td>
    <td></td>
    <td></td>
  </tr>
  <tr>
    <td>5</td>
    <td></td>
    <td></td>
    <td></td>
    <td></td>
  </tr>
    </tbody>
</table>

# Imports

In [5]:
# ============================================================
# 1. Imports
# Standard library
# ============================================================

import json
import urllib.request
from time import sleep

# ============================================================
# 2. Scientific Python stack
# ============================================================

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.optimize import fsolve

# ============================================================
# 3. Networking / HTTP utilities
# ============================================================

import requests

# ============================================================
# 4. QKIT modules
# ============================================================

import qkit
from qkit.measure.spectroscopy.spectroscopy import spectrum
from qkit.storage.store import Data
from qkit.analysis.circle_fit.circle_fit_2019 import circuit

ModuleNotFoundError: No module named 'numpy'

# QKIT Configuration

In [ ]:
RUN_ID = "[RUN_ID]"
USER_NAME = "[USER_NAME]"
CHIP_NAME = "[DEVICE_OR_CHIP_NAME]"

qkit.cfg["run_id"] = RUN_ID
qkit.cfg["user"] = USER_NAME
qkit.cfg["load_visa"] = True

qkit.start()

##  Helper Functions for Loading Measurement Files

In [ ]:

def load_h5(date, folder_name):
    """
    Load an HDF5 measurement file from the configured storage path.
    """
    path = "E:/{}/{}/{}.h5".format(date, folder_name, folder_name)
    return Data(path)


def load_txt(date, folder_name):
    """
    Load a text-based measurement file from the configured storage path.
    """
    path = "D:/{}/{}/{}.txt".format(date, folder_name, folder_name)
    return np.loadtxt(path)

## Helper Functions for Temperature Readout

In [ ]:
DEVICE_IP = "[DEVICE_IP]"
TEMPERATURE_MONITOR_URL = "[TEMPERATURE_MONITOR_URL]"
TIMEOUT = 10


def get_latest_channel_measurement():
    """
    Read the latest measurement data from the configured device endpoint.
    """
    url = "http://{}:5001/channel/measurement/latest".format(DEVICE_IP)
    response = requests.get(url, timeout=TIMEOUT)
    return response.json()


def get_temperature_mxc():
    """
    Read the MXC temperature from the configured thermometer endpoint.
    """
    response = urllib.request.urlopen(TEMPERATURE_MONITOR_URL)
    data = json.loads(response.read())
    return data[1]["MC_2"]["T"]

# Plotting Configuration

In [ ]:
mpl.rcParams["figure.dpi"] = 150

# Instrument configuration

## VNA Config

In [ ]:
VNA_NAME = "vna"
VNA_DRIVER = "[VNA_DRIVER]"
VNA_ADDRESS = "[VNA_ADDRESS]"

In [ ]:
#Initialize instruments
vna = qkit.instruments.create(
    VNA_NAME,
    VNA_DRIVER,
    address=VNA_ADDRESS,
)
m = spectrum(vna=vna)

## Current source Config

In [ ]:
CURRENT_SOURCE_NAME = "current"
CURRENT_SOURCE_DRIVER = "Yokogawa_GS200"
CURRENT_SOURCE_ADDRESS = "[CURRENT_SOURCE_ADDRESS]"

In [ ]:
#Initialize instruments
yoko = qkit.instruments.create(
    CURRENT_SOURCE_NAME,
    CURRENT_SOURCE_DRIVER,
    address=CURRENT_SOURCE_ADDRESS,
)

In [ ]:
#Current-source helper functions
def ramp_current(current_A):
    """
    Ramp the current source to the requested current value.
    """
    yoko.ramp_current(current_A, 5e-5, 0.05, True)
    sleep(0.1)

In [ ]:
# Initialize current source state
ramp_current(0.0e-3)

# Increase the source range beyond the default 10 mA range
yoko.set_source_range(100e-3)

# Optional: enable output when required
# yoko.set_output("on")

# Optional: disable output after measurement
# yoko.set_output("off")

## AnaPico Cofig

In [ ]:
# AnaPico configuration
ANAPICO_NAME = "anapico"
ANAPICO_DRIVER = "[ANAPICO_DRIVER]"
ANAPICO_ADDRESS = "[ANAPICO_ADDRESS]"

In [ ]:
# Initialize instrument
anapico = qkit.instruments.create(
    ANAPICO_NAME,
    ANAPICO_DRIVER,
    address=ANAPICO_ADDRESS,
)

In [1]:
# Optional: set output power
# anapico.set_ch_power(0)

# Optional: set CW frequency
# anapico.set_ch_frequency(5e9)

## MWPower Config

In [2]:
# MW power source configuration
MW_POWER_NAME = "mw_drive"
MW_POWER_DRIVER = "[MW_POWER_DRIVER]"
MW_POWER_ADDRESS = "[MW_POWER_ADDRESS]"

In [ ]:
# Initialize instrument
mw_power = qkit.instruments.create(
    MW_POWER_NAME,
    MW_POWER_DRIVER,
    address=MW_POWER_ADDRESS,
)

In [3]:
# Optional: set output frequency
# mw_power.set_frequency(6.86e9)

# Optional: set output power
# mw_power.set_power(7)

# Optional: enable output during measurement
# mw_power.set_status(True)

# Optional: disable output after measurement
# mw_power.set_status(False)

# VNA Parameters

In [ ]:
vna.set_port1_edel(46.084e-9)

In [ ]:
vna.get_port1_edel()

In [ ]:
print(vna.get_centerfreq())
print(vna.get_span())
print(vna.get_power())
print(vna.get_bandwidth())
print(vna.get_aver)ages())
print(vna.get_bandwidth())
print(vna.get_nop())
print(vna.get_sweeptime_averages())

# Overview trace

In [ ]:
vna.set_startfreq(5.0e9)
vna.set_stopfreq(6.1e9)
vna.set_port1_edel(31.1e-9)
vna.set_nop(11001)
vna.set_power(-30.0)
vna.set_bandwidth(1000)
vna.set_averages(2)
vna.set_Average(True)

print(vna.get_sweeptime())
print(vna.get_sweeptime_averages())

In [ ]:
m.dirname = "overviewTrace_{:g}dBm_{:g}-{:g}GHz_{}nop_{:g}kHzIF_{}avgs".format(
    vna.get_power(), vna.get_startfreq()/1e9, vna.get_stopfreq()/1e9, vna.get_nop(), vna.get_bandwidth()/1e3, vna.get_averages()
)
m.measure_1D()

# TimeSweep

In [ ]:
def do_nothing(x):
    pass

In [ ]:
vna.set_centerfreq(fr)
vna.set_port1_edel(edel_rs)
vna.set_span(100.0e6)
vna.set_nop(501)
vna.set_power(-60.0)
vna.set_bandwidth(500)
vna.set_averages(3)
vna.set_Average(False)

print(vna.get_sweeptime())
print(vna.get_sweeptime_averages())

In [ ]:
m.dirname = "TimeSweep_{}_{:g}dBm_{:g}-{:g}GHz_{}nop_{:g}kHzIF_{}avgs".format(
    resonator, vna.get_power(), vna.get_startfreq()/1e9, vna.get_stopfreq()/1e9, vna.get_nop(), vna.get_bandwidth()/1e3, vna.get_averages()
)

N = 4000
m.set_x_parameters(np.linspace(1, N, N), "measurment no.", do_nothing, "")
m.measure_2D()

# Resonators

In [ ]:
# set the total number of resonances found
# set a standard port delay
# set the frequencies of the resonances found manually on VNA
# choose an appropriate span for each
# if necessary change the port delay

resonance_totalnumber = 2




# Define the resonance parameters in a dictionary
resonance_params = {
    # 0dB on VNA #power = -50dB
    1: {"frequency": 7.981232e9, "span": 7e6, "edel": 46.084e-9},
    # 0dB on VNA #power = -30dB
    2: {"frequency": 8.737700e9, "span": 2e6, "edel": 46.084e-9},
}


In [ ]:
# use this cell to view each resonance on the VNA
# Set VNA settings for resonance number n

resonance_number = 2
params = resonance_params[resonance_number]

vna.set_power(-30)
vna.set_averages(5)
vna.set_bandwidth(500)
vna.set_Average(True)
vna.set_nop(501)

vna.set_centerfreq(params["frequency"])
vna.set_span(params["span"])
vna.set_port1_edel(params["edel"])


In [ ]:
# Auto scale traces on VNA graph
vna.write(':DISP:WIND1:TRAC1:Y:AUTO')
vna.write(':DISP:WIND1:TRAC2:Y:AUTO')

# Power Sweep

## Definitions

In [ ]:
# Define dynamic power function
def dynamic_power(p):
    vna.set_power(p)
    powerratio = 4**(0.1*(p - start_power))  # p1/p2
    vna.set_averages(max(3, round(start_averages / powerratio, 0)))

# Define dynamic power averages function
def dynamic_power_avgs(p):
    powerratio = 4**(0.1*(p - start_power))  # p1/p2
    return np.maximum(3, np.round(start_averages / powerratio, 0))


## PowerSetup

In [ ]:
resonance_params = {
    
    1: {"frequency": 7.254622e9, "span": 12e6, "edel": 40.203e-9, "bandwidth": 80, "nop": 201},
    2: {"frequency": 7.025071e9, "span": 300e3, "edel": 40.168e-9, "bandwidth": 80, "nop": 201},
 
}


In [ ]:
# Initialize array with powers to measure at and set initial averaging

start_averages = 1024   
start_power = -80
mid_power = -40
stop_power = 0
delta_p1 = 5
delta_p2 = 2

powers1 = np.linspace(start_power, mid_power, int(round(abs((start_power - mid_power)/delta_p1),0)) + 1)
powers2 = np.linspace(mid_power+delta_p2, stop_power, int(round(abs(((mid_power+delta_p2) - stop_power)/delta_p2),0)) + 1)

powers = np.concatenate((powers1,powers2),axis=0)

print(powers)

print(dynamic_power_avgs(powers))

# expected time in minutes:
print(vna.get_sweeptime()*np.sum(dynamic_power_avgs(powers))/60)



In [ ]:
# Loop through resonances
for resonance_number, params in resonance_params.items():
    if resonance_number in [3, 5, 6]:
        # Perform power sweep measurement
        vna.set_power(powers[0])
        vna.set_averages(start_averages)
        vna.set_Average(True)
        

        vna.set_centerfreq(params["frequency"])
        vna.set_span(params["span"])
        vna.set_port1_edel(params["edel"])
        vna.set_bandwidth(params["bandwidth"]) 
        vna.set_nop(params["nop"])  

        m.set_x_parameters(powers, "power", dynamic_power, "dBm")
        m.set_resonator_fit(True, "circle_fit_reflection")

        req = requests.get(url, timeout=TIMEOUT)
        data = req.json()
        mxc_temperature = data['temperature'] * 1000

        m.dirname = "{}_powersweep_{}_{:g}-{:g}dBm_{:g}-{:g}GHz_temp_{:g}mK_{}nop_{:g}kHzIF_dynamicAvg".format(
            resonance_number, chip_name, powers[0], powers[-1], vna.get_startfreq()/1e9, vna.get_stopfreq()/1e9, mxc_temperature, vna.get_nop(), vna.get_bandwidth()/1e3
        )
        m.comment = "with 1x -20dB attenuator on VNA_input"
        
        m.measure_2D()
        m.set_resonator_fit(False, "circle_fit_reflection")
        
vna.hold(True)

# SelfKerrCoefficient

## Measurements

In [ ]:
# Loop through resonances
for resonance_number, params in resonance_params.items():
    if resonance_number in [3, 5, 6]:
        # Perform power sweep measurement
        vna.set_power(-20)
        vna.set_averages(5)  # Set constant averages to 5
        vna.set_Average(True)

        vna.set_centerfreq(params["frequency"])
        vna.set_span(params["span"])
        vna.set_port1_edel(params["edel"])
        vna.set_bandwidth(params["bandwidth"]) 
        vna.set_nop(params["nop"])  

        m.set_x_parameters(np.arange(-20, 12, 2), "power")  # Powers from -20 dBm to 10 dBm with steps of 2 dBm
        m.set_resonator_fit(True, "circle_fit_reflection")

        req = requests.get(url, timeout=TIMEOUT)
        data = req.json()
        mxc_temperature = data['temperature'] * 1000

        m.dirname = "{}_powersweep_{}_{:g}-{:g}dBm_{:g}-{:g}GHz_temp_{:g}mK_{}nop_{:g}kHzIF_constantAvg".format(
            resonance_number, chip_name, -20, 10, vna.get_startfreq()/1e9, vna.get_stopfreq()/1e9, mxc_temperature, vna.get_nop(), vna.get_bandwidth()/1e3
        )
        m.comment = "with 0dB attenuator on VNA"
        m.measure_2D()
        m.set_resonator_fit(False, "circle_fit_reflection")
        
vna.hold(True)

## MonitorOverTime

In [ ]:
# Set up VNA parameters
vna.set_span(0)
vna.set_nop(2)
vna.set_power(-70)
vna.set_bandwidth(70)
vna.set_averages(10)
vna.set_Average(True)

# Print sweep times
print("Single sweep time:", vna.get_sweeptime())
print("Averaged sweep time:", vna.get_sweeptime_averages())

# Set up microwave power source
mw_power.set_power(-5)

# Frequency sweep parameters
start_freq = 3.10e9
stop_freq = 3.40e9
delta_f = 2.0e6
sweep_freqs = np.linspace(start_freq, stop_freq, int((stop_freq - start_freq) / delta_f) + 1)

rep = np.arange(100)

m.set_x_parameters(rep, 'repetition', do_nothing, x_unit='#')
m.set_y_parameters(sweep_freqs, 'drive_freq', mw_power.set_frequency, y_unit = 'Hz')

m.dirname = "{}_TwoToneMonitor_{}_{:g}-{:g}GHz_{:g}x".format(
    resonance_number, chip_name, sweep_freqs[0]/1e9, sweep_freqs[-1]/1e9, rep[-1]
)

m.comment = "with 0dB attenuator for qubit drive"

# Perform measurement
mw_power.set_status(True)
m.measure_3D()
mw_power.set_status(False)

# FluxSweep

## Flux-Sweep Current-Source 

In [ ]:
yoko.set_output("on")

CURRENT_MIN_A = -10e-3
CURRENT_MAX_A = 10e-3
N_CURRENT_POINTS = 11
current_values = np.linspace(CURRENT_MIN_A, CURRENT_MAX_A, N_CURRENT_POINTS)

## Coil Model and Flux-Conversion Helpers

In [ ]:
class Coil:
    def __init__(self, radius_m, z0_m, n_turns):
        self.radius_m = radius_m
        self.z0_m = z0_m
        self.n_turns = n_turns

    def B_z(self, z_m, current_A):
        """
        Calculate the magnetic field at position z_m along the coil axis.
        """
        return (
            self.n_turns
            * sc.mu_0
            * current_A
            * self.radius_m**2
            / (self.radius_m**2 + (z_m - self.z0_m) ** 2) ** (3 / 2)
        )

    def flux(self, magnetic_field_T):
        """
        Calculate the magnetic flux through the coil area.
        """
        return magnetic_field_T * np.pi * self.radius_m**2


# Coil parameters (template values; replace as needed)
coil_1 = Coil(radius_m=17.5e-3, z0_m=17.5e-3, n_turns=240)
coil_2 = Coil(radius_m=17.5e-3, z0_m=-17.5e-3, n_turns=280)

# Magnetic flux quantum
PHI_0 = sc.h / (2 * sc.e)

In [ ]:
# Run flux-sweep measurement
for resonance_number, params in resonance_params.items():
    if resonance_number in [3, 5, 6]:
        # VNA settings
        vna.set_power(-30.0)
        vna.set_averages(20)
        vna.set_Average(True)

        vna.set_centerfreq(params["frequency"])
        vna.set_span(params["span"])
        vna.set_port1_edel(params["edel"])
        vna.set_bandwidth(params["bandwidth"])
        vna.set_nop(params["nop"])

        # Measurement settings
        m.set_x_parameters(current_values, "current", ramp_current, "A")
        m.set_resonator_fit(True, "circle_fit_reflection")

        # Optional temperature readout
        # req = requests.get(url, timeout=TIMEOUT)
        # data = req.json()
        # mxc_temperature = data["temperature"] * 1000

        m.dirname = "{}_FluxSweep_{}_{:g}-{:g}A_{:g}-{:g}GHz_temp_{:g}mK_{}nop_{:g}kHzIF".format(
            resonance_number,
            CHIP_NAME,
            CURRENT_MIN_A,
            CURRENT_MAX_A,
            vna.get_startfreq() / 1e9,
            vna.get_stopfreq() / 1e9,
            mxc_temperature,
            vna.get_nop(),
            vna.get_bandwidth() / 1e3,
        )

        m.comment = "[MEASUREMENT_COMMENT]"
        m.measure_2D()

        m.set_resonator_fit(False, "circle_fit_reflection")